<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_04_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_04 – Dataset preparation**


# **Introducción**

Luego del proceso de selección de features, el problema ha sido transformado desde una serie de tiempo cruda hacia un dataset tabular compuesto por indicadores técnicos y variables contextuales. Estos features ya incorporan información temporal, de tendencia, momentum, reversión y volatilidad, reduciendo la necesidad de que el modelo aprenda directamente dependencias secuenciales complejas.

En este contexto, el problema deja de ser puramente secuencial y pasa a ser un problema de regresión tabular con señal débil y ruido elevado, típico en aplicaciones de mercados financieros intradía.

## **Objetivo**

El objetivo es entrenar modelos capaces de:

- capturar relaciones no lineales entre indicadores y el target
- ser robustos al ruido y a la baja señal
- generalizar correctamente en datos out-of-sample
- mantener estabilidad entre distintos períodos y regímenes de mercado

Dado este escenario, se priorizan modelos que han demostrado buen desempeño en problemas tabulares con estas características.


## **Justificación del uso de modelos tabulares**

La elección de modelos tabulares se basa en los siguientes puntos:

- los indicadores técnicos ya resumen la información temporal relevante
- no se observa una dependencia secuencial compleja que justifique modelos recurrentes o transformers
- los modelos tabulares son más robustos en presencia de señal débil
- permiten interpretar mejor la contribución de cada feature
- requieren menor complejidad computacional y de tuning

En consecuencia, se descarta el uso prioritario de modelos como LSTM o Transformers, ya que no aportan ventajas claras en este contexto.

## **Modelos a entrenar**

Se define el siguiente conjunto de modelos:

- Modelos lineales (baseline):
  - Ridge Regression

- Modelos de ensamble:
  - Random Forest

- Modelos de boosting (principales):
  - XGBoost
  - LightGBM

- Modelos no lineales simples (opcional):
  - MLP (Multi-Layer Perceptron)

## **Estrategia**














El enfoque de entrenamiento será progresivo:

1. Establecer un baseline con modelos lineales
2. Evaluar modelos de ensamble para capturar no linealidades
3. Priorizar modelos de boosting como candidatos principales
4. Comparar desempeño en métricas out-of-sample
5. Validar robustez y estabilidad de los resultados

Este enfoque permite identificar de forma clara si la complejidad adicional de los modelos se traduce en mejoras reales en capacidad predictiva.

# **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator



In [4]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_DELTA_60 = Path(os.environ.get("IN_PARQUET_DELTA_60", "data/features/mnq_delta_60.parquet"))
IN_PARQUET_DELTA_90 = Path(os.environ.get("IN_PARQUET_DELTA_90", "data/features/mnq_delta_90.parquet"))

##############

OUT_SPLITS = Path(os.environ.get("OUT_PARQUET", "data/splits/splits.json"))


# PARA EL NOTEBOOK:

IN_PARQUET_DELTA_60 = DRIVE_DIR / IN_PARQUET_DELTA_60
IN_PARQUET_DELTA_90 = DRIVE_DIR / IN_PARQUET_DELTA_90






# **1. Carga de datos**

## **1.1. Carga de dataset `mnq_delta_*.parquet`**




In [5]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_mnq_parquet(path: Path):
    os.path.exists(path)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(path)
    return mnq_parquet

## **1.2. Información de dataset `mnq_*`**



In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")



## **1.3. Eliminación de OHLCV**

In [7]:
def delete_ohlcv(df, target_col):
    cols_to_drop = ["open", "high", "low", "close", "volume"]

    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df

## **1.3. Carga de mnq e información**




In [8]:
REGIME_ID = '''
regime_id = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}
'''

In [9]:
mnq_delta_60 = load_mnq_parquet(IN_PARQUET_DELTA_60)
mnq_delta_60 = delete_ohlcv(mnq_delta_60, "delta_60")
info_mnq_delta_60 = mnq_dataset_info(mnq_delta_60, name="mnq_delta_60", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_60)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_60
Shape: (700595, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [10]:
mnq_delta_90 = load_mnq_parquet(IN_PARQUET_DELTA_90)
mnq_delta_90 = delete_ohlcv(mnq_delta_90, "delta_90")
info_mnq_delta_90 = mnq_dataset_info(mnq_delta_90, name="mnq_delta_90", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_90)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_90
Shape: (700595, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


## **1.4. Filtrado de régimen**


In [11]:
import pandas as pd


def filter_intraday_regimes(
    df: pd.DataFrame,
    *,
    regime_col: str = "regime_id",
    date_col: str | None = "date",
    minute_col: str | None = "minute_of_day",
    allowed_regimes: tuple[int, ...] = (1,), #(1, 2, 3),
    validate_sorted: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filtra un dataset intradía eliminando overnight y closing,
    manteniendo solo los regímenes especificados.

    Además:
    - verifica orden temporal
    - asegura consistencia básica

    Retorna
    -------
    df_filtered : pd.DataFrame
    """

    if regime_col not in df.columns:
        raise ValueError(f"No existe la columna {regime_col}")

    # -------------------------
    # Filtrado
    # -------------------------
    df_filtered = df[df[regime_col].isin(allowed_regimes)].copy()

    if verbose:
        print(f"\n=== FILTER REGIMES ===")
        print(f"Original shape: {df.shape}")
        print(f"Filtered shape: {df_filtered.shape}")
        print(f"Regimes kept: {allowed_regimes}")
        print("Distribución:")
        print(df_filtered[regime_col].value_counts().sort_index())

    # -------------------------
    # Orden temporal
    # -------------------------
    if validate_sorted:
        if date_col and minute_col and date_col in df_filtered.columns and minute_col in df_filtered.columns:
            sorted_ok = (
                df_filtered
                .sort_values([date_col, minute_col])
                .index.equals(df_filtered.index)
            )

            if not sorted_ok:
                print("⚠️ Dataset NO estaba ordenado → reordenando...")
                df_filtered = df_filtered.sort_values([date_col, minute_col])

        else:
            if not df_filtered.index.is_monotonic_increasing:
                print("⚠️ Index no ordenado → reordenando...")
                df_filtered = df_filtered.sort_index()

    # -------------------------
    # Validaciones finales
    # -------------------------
    if df_filtered.empty:
        raise ValueError("Dataset vacío después del filtrado")

    if df_filtered.isna().any().any():
        print("⚠️ WARNING: Hay NaNs en el dataset filtrado")

    return df_filtered

In [12]:
mnq_delta_60_filtered = filter_intraday_regimes(mnq_delta_60)
info_mnq_delta_60_filtered = mnq_dataset_info(mnq_delta_60_filtered, name="mnq_delta_60_filtered", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_60_filtered)



=== FILTER REGIMES ===
Original shape: (700595, 9)
Filtered shape: (77700, 9)
Regimes kept: (1,)
Distribución:
regime_id
1    77700
Name: count, dtype: int64
Dataset: mnq_delta_60_filtered
Shape: (77700, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 08:30:00-05:00  ->  2025-06-13 09:29:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 510, 'max_minute_of_day': 569}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 13:30:00+00:00  ->  2025-06-13 13:29:00+00:00


In [13]:
mnq_delta_90_filtered = filter_intraday_regimes(mnq_delta_90)
info_mnq_delta_90_filtered = mnq_dataset_info(mnq_delta_90_filtered, name="mnq_delta_90_filtered", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_90_filtered)


=== FILTER REGIMES ===
Original shape: (700595, 9)
Filtered shape: (77700, 9)
Regimes kept: (1,)
Distribución:
regime_id
1    77700
Name: count, dtype: int64
Dataset: mnq_delta_90_filtered
Shape: (77700, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 08:30:00-05:00  ->  2025-06-13 09:29:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 510, 'max_minute_of_day': 569}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 13:30:00+00:00  ->  2025-06-13 13:29:00+00:00


In [14]:
mnq_delta_60 = mnq_delta_60_filtered.copy()
mnq_delta_90 = mnq_delta_90_filtered.copy()

# **2. Separación de variables (X e y)**

En este punto del pipeline, el dataset ya contiene features diseñadas (indicadores técnicos y variables contextuales) y un target definido (`delta_60` o `delta_90`).

El objetivo ahora es **formalizar el problema de aprendizaje supervisado**, separando:

* **X (features):** variables explicativas que contienen la señal
* **y (target):** variable a predecir

Esta separación es necesaria porque los modelos de machine learning aprenden una función:

$$
f(X) \rightarrow y
$$

donde cada fila representa una observación independiente.

Es importante destacar que, aunque el dataset proviene de una serie temporal, en este enfoque tabular:

* la dependencia temporal ya está incorporada en los features (EMA, ROC, etc.)
* cada fila puede tratarse como una observación independiente
* la coherencia temporal se mantiene a través del índice datetime

Durante este paso, algunas columnas auxiliares (como `date`) se conservan en `X` **únicamente para facilitar el split temporal posterior**, pero no formarán parte del entrenamiento del modelo.


## **2.1. Código — Separación de X e y**

In [15]:
def separate_predictors_and_target(df, target_col):
    # asegurar orden temporal
    df = df.sort_index().copy()

    # target
    y = df[target_col].copy()

    # features (incluye columnas auxiliares como 'date')
    X = df.drop(columns=[target_col]).copy()

    # validación de alineación
    assert (X.index == y.index).all(), "X e y no están alineados"

    return X, y

In [16]:
X_60, y_60 = separate_predictors_and_target(
    mnq_delta_60,
    target_col="delta_60"
)

In [17]:
X_90, y_90 = separate_predictors_and_target(
    mnq_delta_90,
    target_col="delta_90"
)

**Resultado esperado**

* `X`: todas las columnas excepto el target (incluye `date` por ahora)
* `y`: serie con el target
* ambos perfectamente alineados por el índice datetime

# **3. Split temporal del dataset**

Una vez separados `X` e `y`, el siguiente paso es dividir el dataset en **train, valid y test**, respetando estrictamente el orden temporal.

A diferencia de problemas tradicionales, en datos financieros no se puede hacer un split aleatorio. La división debe realizarse por **días completos y en orden cronológico**, de modo que:

* el modelo se entrene solo con información pasada
* se evalúe sobre datos futuros (out-of-sample)
* se evite cualquier forma de *data leakage*

Para ello, se utilizan los días únicos del dataset y se asignan en proporción **70% / 15% / 15%** para train, valid y test, respectivamente.

Este enfoque garantiza una evaluación realista del modelo en condiciones similares a las de producción.


## **3.1. Código - Split temporal**

### **1. Validación de orden y alineación temporal**

El primero paso es comprobar la alineación temporal correcta:

In [18]:
def validate_X_y_alignment(X, y):
    if not X.index.is_monotonic_increasing:
        raise ValueError("X no está ordenado cronológicamente por índice.")

    if not y.index.is_monotonic_increasing:
        raise ValueError("y no está ordenado cronológicamente por índice.")

    if not X.index.equals(y.index):
        raise ValueError("X e y no tienen el mismo índice; no están alineados.")

    print("Validación OK: X e y están ordenados y alineados temporalmente.")
    print(f"Filas X: {len(X)} | Filas y: {len(y)}")
    print(f"Rango temporal: {X.index.min()} -> {X.index.max()}")

In [19]:
validate_X_y_alignment(X_60, y_60)

Validación OK: X e y están ordenados y alineados temporalmente.
Filas X: 77700 | Filas y: 77700
Rango temporal: 2020-01-02 08:30:00-05:00 -> 2025-06-13 09:29:00-04:00


In [20]:
validate_X_y_alignment(X_90, y_90)

Validación OK: X e y están ordenados y alineados temporalmente.
Filas X: 77700 | Filas y: 77700
Rango temporal: 2020-01-02 08:30:00-05:00 -> 2025-06-13 09:29:00-04:00


### **2. Obtención de días únicos**

In [21]:
def get_unique_trading_days(X, date_col="date"):
    if date_col not in X.columns:
        raise ValueError(f"No existe la columna '{date_col}' en X.")

    unique_days = pd.Index(pd.to_datetime(X[date_col]).dt.normalize().unique()).sort_values()

    print(f"Días únicos: {len(unique_days)}")
    print(f"Primer día: {unique_days.min().date()}")
    print(f"Último día: {unique_days.max().date()}")

    return unique_days

In [22]:
unique_days_60 = get_unique_trading_days(X_60, date_col="date")

Días únicos: 1295
Primer día: 2020-01-02
Último día: 2025-06-13


In [23]:
unique_days_90 = get_unique_trading_days(X_90, date_col="date")

Días únicos: 1295
Primer día: 2020-01-02
Último día: 2025-06-13


### **3. Dividir días train, valid y test**

In [24]:
def split_days_train_valid_test(unique_days, train_ratio=0.70, valid_ratio=0.15):
    n_days = len(unique_days)

    n_train = int(n_days * train_ratio)
    n_valid = int(n_days * valid_ratio)
    n_test = n_days - n_train - n_valid

    train_days = unique_days[:n_train]
    valid_days = unique_days[n_train:n_train + n_valid]
    test_days  = unique_days[n_train + n_valid:]

    print(f"Train days: {len(train_days)} | {train_days.min().date()} -> {train_days.max().date()}")
    print(f"Valid days: {len(valid_days)} | {valid_days.min().date()} -> {valid_days.max().date()}")
    print(f"Test days : {len(test_days)} | {test_days.min().date()} -> {test_days.max().date()}")

    return train_days, valid_days, test_days

In [25]:
train_days_60, valid_days_60, test_days_60 = split_days_train_valid_test(
    unique_days_60,
    train_ratio=0.70,
    valid_ratio=0.15,
)

Train days: 906 | 2020-01-02 -> 2023-10-30
Valid days: 194 | 2023-10-31 -> 2024-08-21
Test days : 195 | 2024-08-22 -> 2025-06-13


In [26]:
train_days_90, valid_days_90, test_days_90 = split_days_train_valid_test(
    unique_days_90,
    train_ratio=0.70,
    valid_ratio=0.15,
)

Train days: 906 | 2020-01-02 -> 2023-10-30
Valid days: 194 | 2023-10-31 -> 2024-08-21
Test days : 195 | 2024-08-22 -> 2025-06-13


### **4. Aplicar split sobre X e y**

In [27]:
def apply_day_split(X, y, train_days, valid_days, test_days, date_col="date"):
    x_days = pd.to_datetime(X[date_col]).dt.normalize()

    train_mask = x_days.isin(train_days)
    valid_mask = x_days.isin(valid_days)
    test_mask  = x_days.isin(test_days)

    X_train, y_train = X.loc[train_mask].copy(), y.loc[train_mask].copy()
    X_valid, y_valid = X.loc[valid_mask].copy(), y.loc[valid_mask].copy()
    X_test,  y_test  = X.loc[test_mask].copy(),  y.loc[test_mask].copy()

    print(f"Train -> X: {X_train.shape} | y: {y_train.shape}")
    print(f"Valid -> X: {X_valid.shape} | y: {y_valid.shape}")
    print(f"Test  -> X: {X_test.shape} | y: {y_test.shape}")

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [28]:
X_train_60, X_valid_60, X_test_60, y_train_60, y_valid_60, y_test_60 = apply_day_split(
    X_60,
    y_60,
    train_days_60,
    valid_days_60,
    test_days_60,
    date_col="date",
)

Train -> X: (54360, 8) | y: (54360,)
Valid -> X: (11640, 8) | y: (11640,)
Test  -> X: (11700, 8) | y: (11700,)


In [29]:
X_train_90, X_valid_90, X_test_90, y_train_90, y_valid_90, y_test_90 = apply_day_split(
    X_90,
    y_90,
    train_days_90,
    valid_days_90,
    test_days_90,
    date_col="date",
)

Train -> X: (54360, 8) | y: (54360,)
Valid -> X: (11640, 8) | y: (11640,)
Test  -> X: (11700, 8) | y: (11700,)


### **5. Validar no solapamiento**

In [30]:
def validate_no_day_overlap(train_days, valid_days, test_days):
    train_set = set(train_days)
    valid_set = set(valid_days)
    test_set = set(test_days)

    if train_set & valid_set:
        raise ValueError("Hay solapamiento entre train y valid.")
    if train_set & test_set:
        raise ValueError("Hay solapamiento entre train y test.")
    if valid_set & test_set:
        raise ValueError("Hay solapamiento entre valid y test.")

    print("Validación OK: no hay solapamiento entre días de train, valid y test.")

In [31]:
validate_no_day_overlap(train_days_60, valid_days_60, test_days_60)

Validación OK: no hay solapamiento entre días de train, valid y test.


In [32]:
validate_no_day_overlap(train_days_90, valid_days_90, test_days_90)

Validación OK: no hay solapamiento entre días de train, valid y test.


### **6. Eliminar columna 'date'**

In [33]:
def drop_date_column(X, cols_to_drop=None):
    if cols_to_drop is None:
        cols_to_drop = ["date"]

    X_clean = X.drop(columns=[c for c in cols_to_drop if c in X.columns]).copy()

    print(f"Shape original: {X.shape}")
    print(f"Shape limpia  : {X_clean.shape}")
    print(f"Columnas finales: {list(X_clean.columns)}")

    return X_clean

In [34]:
X_train_60 = drop_date_column(X_train_60)
X_valid_60 = drop_date_column(X_valid_60)
X_test_60  = drop_date_column(X_test_60)

Shape original: (54360, 8)
Shape limpia  : (54360, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (11640, 8)
Shape limpia  : (11640, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (11700, 8)
Shape limpia  : (11700, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']


In [35]:
X_train_90 = drop_date_column(X_train_90)
X_valid_90 = drop_date_column(X_valid_90)
X_test_90  = drop_date_column(X_test_90)

Shape original: (54360, 8)
Shape limpia  : (54360, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (11640, 8)
Shape limpia  : (11640, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (11700, 8)
Shape limpia  : (11700, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']


### **7. Guardado de splits**

#### **Verificación de orden**

In [36]:
def validate_and_sort_before_save(X, y, split_name="split"):
    if not X.index.equals(y.index):
        raise ValueError(f"{split_name}: X e y no están alineados por índice.")

    X = X.sort_index().copy()
    y = y.sort_index().copy()

    if not X.index.is_monotonic_increasing:
        raise ValueError(f"{split_name}: X no quedó ordenado cronológicamente.")
    if not y.index.is_monotonic_increasing:
        raise ValueError(f"{split_name}: y no quedó ordenado cronológicamente.")

    print(f"{split_name} OK | filas={len(X)} | {X.index.min()} -> {X.index.max()}")

    return X, y

In [37]:
X_train_60, y_train_60 = validate_and_sort_before_save(X_train_60, y_train_60, "train_60")
X_valid_60, y_valid_60 = validate_and_sort_before_save(X_valid_60, y_valid_60, "valid_60")
X_test_60,  y_test_60  = validate_and_sort_before_save(X_test_60,  y_test_60,  "test_60")

X_train_90, y_train_90 = validate_and_sort_before_save(X_train_90, y_train_90, "train_90")
X_valid_90, y_valid_90 = validate_and_sort_before_save(X_valid_90, y_valid_90, "valid_90")
X_test_90,  y_test_90  = validate_and_sort_before_save(X_test_90,  y_test_90,  "test_90")

train_60 OK | filas=54360 | 2020-01-02 08:30:00-05:00 -> 2023-10-30 09:29:00-04:00
valid_60 OK | filas=11640 | 2023-10-31 08:30:00-04:00 -> 2024-08-21 09:29:00-04:00
test_60 OK | filas=11700 | 2024-08-22 08:30:00-04:00 -> 2025-06-13 09:29:00-04:00
train_90 OK | filas=54360 | 2020-01-02 08:30:00-05:00 -> 2023-10-30 09:29:00-04:00
valid_90 OK | filas=11640 | 2023-10-31 08:30:00-04:00 -> 2024-08-21 09:29:00-04:00
test_90 OK | filas=11700 | 2024-08-22 08:30:00-04:00 -> 2025-06-13 09:29:00-04:00


#### **Definición de rutas**

In [38]:
from pathlib import Path
import os

##############

OUT_SPLITS = Path(os.environ.get("OUT_SPLITS", "data/splits/splits.json"))

# DELTA 60
OUT_PARQUET_DELTA_60_X_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TRAIN", "data/splits/mnq_delta_60_X_train.parquet"))
OUT_PARQUET_DELTA_60_X_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_VALID", "data/splits/mnq_delta_60_X_valid.parquet"))
OUT_PARQUET_DELTA_60_X_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TEST",  "data/splits/mnq_delta_60_X_test.parquet"))

OUT_PARQUET_DELTA_60_Y_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_TRAIN", "data/splits/mnq_delta_60_y_train.parquet"))
OUT_PARQUET_DELTA_60_Y_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_VALID", "data/splits/mnq_delta_60_y_valid.parquet"))
OUT_PARQUET_DELTA_60_Y_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_TEST",  "data/splits/mnq_delta_60_y_test.parquet"))

# DELTA 90
OUT_PARQUET_DELTA_90_X_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TRAIN", "data/splits/mnq_delta_90_X_train.parquet"))
OUT_PARQUET_DELTA_90_X_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_VALID", "data/splits/mnq_delta_90_X_valid.parquet"))
OUT_PARQUET_DELTA_90_X_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TEST",  "data/splits/mnq_delta_90_X_test.parquet"))

OUT_PARQUET_DELTA_90_Y_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_TRAIN", "data/splits/mnq_delta_90_y_train.parquet"))
OUT_PARQUET_DELTA_90_Y_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_VALID", "data/splits/mnq_delta_90_y_valid.parquet"))
OUT_PARQUET_DELTA_90_Y_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_TEST",  "data/splits/mnq_delta_90_y_test.parquet"))

# INPUTS
IN_PARQUET_DELTA_60 = DRIVE_DIR / IN_PARQUET_DELTA_60
IN_PARQUET_DELTA_90 = DRIVE_DIR / IN_PARQUET_DELTA_90

# OUTPUTS DELTA 60
OUT_PARQUET_DELTA_60_X_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TRAIN
OUT_PARQUET_DELTA_60_X_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_VALID
OUT_PARQUET_DELTA_60_X_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TEST

OUT_PARQUET_DELTA_60_Y_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_TRAIN
OUT_PARQUET_DELTA_60_Y_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_VALID
OUT_PARQUET_DELTA_60_Y_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_TEST

# OUTPUTS DELTA 90
OUT_PARQUET_DELTA_90_X_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TRAIN
OUT_PARQUET_DELTA_90_X_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_VALID
OUT_PARQUET_DELTA_90_X_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TEST

OUT_PARQUET_DELTA_90_Y_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_TRAIN
OUT_PARQUET_DELTA_90_Y_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_VALID
OUT_PARQUET_DELTA_90_Y_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_TEST

#### **Guardado de splits**

In [39]:
def save_splits_to_parquet(
    X_train, X_valid, X_test,
    y_train, y_valid, y_test,
    paths_dict
):
    # crear directorios si no existen
    for p in paths_dict.values():
        p.parent.mkdir(parents=True, exist_ok=True)

    # guardar X
    X_train.to_parquet(paths_dict["X_train"])
    X_valid.to_parquet(paths_dict["X_valid"])
    X_test.to_parquet(paths_dict["X_test"])

    # guardar y
    y_train.to_frame(name="target").to_parquet(paths_dict["y_train"])
    y_valid.to_frame(name="target").to_parquet(paths_dict["y_valid"])
    y_test.to_frame(name="target").to_parquet(paths_dict["y_test"])

    print("Guardado OK:")
    for k, v in paths_dict.items():
        print(f"{k}: {v}")

In [40]:
save_splits_to_parquet(
    X_train_60, X_valid_60, X_test_60,
    y_train_60, y_valid_60, y_test_60,
    {
        "X_train": OUT_PARQUET_DELTA_60_X_TRAIN,
        "X_valid": OUT_PARQUET_DELTA_60_X_VALID,
        "X_test":  OUT_PARQUET_DELTA_60_X_TEST,
        "y_train": OUT_PARQUET_DELTA_60_Y_TRAIN,
        "y_valid": OUT_PARQUET_DELTA_60_Y_VALID,
        "y_test":  OUT_PARQUET_DELTA_60_Y_TEST,
    }
)

Guardado OK:
X_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_train.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_valid.parquet
X_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_test.parquet
y_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_train.parquet
y_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_valid.parquet
y_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_test.parquet


In [41]:
save_splits_to_parquet(
    X_train_90, X_valid_90, X_test_90,
    y_train_90, y_valid_90, y_test_90,
    {
        "X_train": OUT_PARQUET_DELTA_90_X_TRAIN,
        "X_valid": OUT_PARQUET_DELTA_90_X_VALID,
        "X_test":  OUT_PARQUET_DELTA_90_X_TEST,
        "y_train": OUT_PARQUET_DELTA_90_Y_TRAIN,
        "y_valid": OUT_PARQUET_DELTA_90_Y_VALID,
        "y_test":  OUT_PARQUET_DELTA_90_Y_TEST,
    }
)

Guardado OK:
X_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_train.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_valid.parquet
X_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_test.parquet
y_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_train.parquet
y_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_valid.parquet
y_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_test.parquet


# **4. Escalado de features para modelos sensibles a la escala**

Una vez definidos y guardados los subconjuntos train, valid y test, el siguiente paso es preparar una versión escalada de las variables predictoras para aquellos modelos que dependen de la magnitud de los inputs.

El escalado no es necesario para todos los algoritmos. Modelos basados en árboles, como Random Forest, XGBoost o LightGBM, suelen ser invariantes a la escala de las features. En cambio, modelos lineales regularizados y redes neuronales, como Ridge y MLP, sí se benefician de trabajar con variables centradas y comparables en magnitud.

Para evitar data leakage, el escalador debe ajustarse exclusivamente con el conjunto de entrenamiento y luego aplicarse, sin recalibración, sobre valid y test. De esta forma, se preserva la causalidad temporal y se garantiza una evaluación fuera de muestra consistente.

## **4.1. Aplicación de escalador**

In [42]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def fit_and_apply_standard_scaler(X_train, X_valid, X_test):
    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        index=X_train.index,
        columns=X_train.columns,
    )

    X_valid_scaled = pd.DataFrame(
        scaler.transform(X_valid),
        index=X_valid.index,
        columns=X_valid.columns,
    )

    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        index=X_test.index,
        columns=X_test.columns,
    )

    print("Escalado OK")
    print(f"Train: {X_train_scaled.shape}")
    print(f"Valid: {X_valid_scaled.shape}")
    print(f"Test : {X_test_scaled.shape}")

    return X_train_scaled, X_valid_scaled, X_test_scaled, scaler

In [43]:
X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled, scaler_60 = fit_and_apply_standard_scaler(
    X_train_60,
    X_valid_60,
    X_test_60,
)

Escalado OK
Train: (54360, 7)
Valid: (11640, 7)
Test : (11700, 7)


In [44]:
X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled, scaler_90 = fit_and_apply_standard_scaler(
    X_train_90,
    X_valid_90,
    X_test_90,
)

Escalado OK
Train: (54360, 7)
Valid: (11640, 7)
Test : (11700, 7)


## **4.2. Validación de escalamiento**

In [45]:
def validate_scaled_data(X_train_scaled, X_valid_scaled, X_test_scaled, name="dataset"):
    mean_train = X_train_scaled.mean()
    std_train = X_train_scaled.std()

    print(f"=== Validación {name} ===")
    print("Media (train) — debería ~ 0:")
    print(mean_train.round(4))

    print("\nStd (train) — debería ~ 1:")
    print(std_train.round(4))

    if not X_train_scaled.columns.equals(X_valid_scaled.columns) or not X_train_scaled.columns.equals(X_test_scaled.columns):
        raise ValueError("Columnas inconsistentes entre train/valid/test")

    print("\nColumnas consistentes")
    print(f"Shapes: train={X_train_scaled.shape}, valid={X_valid_scaled.shape}, test={X_test_scaled.shape}")

## **4.3. Guardado de escalamiento**

In [46]:
import joblib

def save_scaled_pipeline(
    X_train_scaled, X_valid_scaled, X_test_scaled,
    scaler,
    paths_dict,
    scaler_path
):
    # crear carpetas
    for p in list(paths_dict.values()) + [scaler_path]:
        p.parent.mkdir(parents=True, exist_ok=True)

    # guardar datasets
    X_train_scaled.to_parquet(paths_dict["X_train"])
    X_valid_scaled.to_parquet(paths_dict["X_valid"])
    X_test_scaled.to_parquet(paths_dict["X_test"])

    # guardar scaler
    joblib.dump(scaler, scaler_path)

    print("\n=== Guardado OK ===")
    for k, v in paths_dict.items():
        print(f"{k}: {v}")

    print(f"Scaler: {scaler_path}")

### **Definición de rutas**

In [47]:
from pathlib import Path
import os

# DELTA 60 ESCALADO
OUT_PARQUET_DELTA_60_X_TRAIN_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TRAIN_SCALED", "data/scaled/mnq_delta_60_X_train_scaled.parquet"))
OUT_PARQUET_DELTA_60_X_VALID_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_VALID_SCALED", "data/scaled/mnq_delta_60_X_valid_scaled.parquet"))
OUT_PARQUET_DELTA_60_X_TEST_SCALED  = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TEST_SCALED",  "data/scaled/mnq_delta_60_X_test_scaled.parquet"))

# DELTA 90 ESCALADO
OUT_PARQUET_DELTA_90_X_TRAIN_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TRAIN_SCALED", "data/scaled/mnq_delta_90_X_train_scaled.parquet"))
OUT_PARQUET_DELTA_90_X_VALID_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_VALID_SCALED", "data/scaled/mnq_delta_90_X_valid_scaled.parquet"))
OUT_PARQUET_DELTA_90_X_TEST_SCALED  = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TEST_SCALED",  "data/scaled/mnq_delta_90_X_test_scaled.parquet"))

OUT_PARQUET_DELTA_60_X_TRAIN_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TRAIN_SCALED
OUT_PARQUET_DELTA_60_X_VALID_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_VALID_SCALED
OUT_PARQUET_DELTA_60_X_TEST_SCALED  = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TEST_SCALED

OUT_PARQUET_DELTA_90_X_TRAIN_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TRAIN_SCALED
OUT_PARQUET_DELTA_90_X_VALID_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_VALID_SCALED
OUT_PARQUET_DELTA_90_X_TEST_SCALED  = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TEST_SCALED

OUT_SCALER_DELTA_60 = Path(os.environ.get("OUT_SCALER_DELTA_60", "data/scaled/scaler_delta_60.pkl"))
OUT_SCALER_DELTA_90 = Path(os.environ.get("OUT_SCALER_DELTA_90", "data/scaled/scaler_delta_90.pkl"))

OUT_SCALER_DELTA_60 = DRIVE_DIR / OUT_SCALER_DELTA_60
OUT_SCALER_DELTA_90 = DRIVE_DIR / OUT_SCALER_DELTA_90

## **4.4. Verificación y guardado final**

In [48]:
validate_scaled_data(X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled, "delta_90")

save_scaled_pipeline(
    X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled,
    scaler_90,
    {
        "X_train": OUT_PARQUET_DELTA_90_X_TRAIN_SCALED,
        "X_valid": OUT_PARQUET_DELTA_90_X_VALID_SCALED,
        "X_test":  OUT_PARQUET_DELTA_90_X_TEST_SCALED,
    },
    OUT_SCALER_DELTA_90
)

=== Validación delta_90 ===
Media (train) — debería ~ 0:
minute_of_day   -0.0
regime_id        0.0
ema_60          -0.0
roc_30          -0.0
roc_60           0.0
stoch_k_20       0.0
atr_norm_10     -0.0
dtype: float64

Std (train) — debería ~ 1:
minute_of_day    1.0
regime_id        0.0
ema_60           1.0
roc_30           1.0
roc_60           1.0
stoch_k_20       1.0
atr_norm_10      1.0
dtype: float64

Columnas consistentes
Shapes: train=(54360, 7), valid=(11640, 7), test=(11700, 7)

=== Guardado OK ===
X_train: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_train_scaled.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_valid_scaled.parquet
X_test: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_test_scaled.parquet
Scaler: /content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl


In [49]:
validate_scaled_data(X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled, "delta_60")

save_scaled_pipeline(
    X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled,
    scaler_60,
    {
        "X_train": OUT_PARQUET_DELTA_60_X_TRAIN_SCALED,
        "X_valid": OUT_PARQUET_DELTA_60_X_VALID_SCALED,
        "X_test":  OUT_PARQUET_DELTA_60_X_TEST_SCALED,
    },
    OUT_SCALER_DELTA_60
)

=== Validación delta_60 ===
Media (train) — debería ~ 0:
minute_of_day   -0.0
regime_id        0.0
ema_60          -0.0
roc_30          -0.0
roc_60           0.0
stoch_k_20       0.0
atr_norm_10     -0.0
dtype: float64

Std (train) — debería ~ 1:
minute_of_day    1.0
regime_id        0.0
ema_60           1.0
roc_30           1.0
roc_60           1.0
stoch_k_20       1.0
atr_norm_10      1.0
dtype: float64

Columnas consistentes
Shapes: train=(54360, 7), valid=(11640, 7), test=(11700, 7)

=== Guardado OK ===
X_train: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_train_scaled.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_valid_scaled.parquet
X_test: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_test_scaled.parquet
Scaler: /content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl


# **5. Predicción de modelos**

Los modelos a entrenar tienen como objetivo predecir una variable continua:

* `delta_60` o `delta_90`

Estas representan el **movimiento futuro del precio** en puntos a un horizonte fijo (60 o 90 minutos).

Por lo tanto, el problema se define como una **regresión supervisada**, donde:

* **Input (`X`)**: indicadores técnicos + variables contextuales
* **Output (`y`)**: retorno futuro (delta)

Cada modelo generará una predicción:

$$
\hat{y}_t = f(X_t)
$$

donde $\hat{y}_t$ es la estimación del movimiento futuro del mercado en ese instante.

---

**Sobre el uso de datos escalados**

No afecta la comparación, y esto es importante entenderlo bien:

* El escalado **solo transforma el espacio de entrada (`X`)**
* El target (`y`) **no se escala**
* Las predicciones siempre están en la **misma unidad original (puntos)**

Por lo tanto:

* MAE, RMSE, R² → comparables ✔️
* Directional accuracy → comparable ✔️

---

**Punto clave**

> Dos modelos pueden usar representaciones distintas de `X` (escalado vs no escalado),
> pero mientras predigan el mismo `y`, la comparación es completamente válida.

---

**Conclusión**

* No hay sesgo en comparar modelos escalados vs no escalados
* Es una práctica estándar en ML tabular
* Lo importante es que todos predicen el mismo target y se evalúan igual

## **5.1. Función unificada de métricas**

Dado que se entrenarán múltiples modelos sobre el mismo problema de regresión (`delta_60` y `delta_90`), resulta conveniente centralizar el proceso de evaluación en una única función estandarizada.

Esta función tendrá como propósito recibir los valores reales (`y_true`) y las predicciones del modelo (`y_pred`), calcular un conjunto consistente de métricas y devolver los resultados en un formato estructurado que facilite su comparación.

La unificación de este proceso permite:

* garantizar que todos los modelos sean evaluados bajo exactamente los mismos criterios
* evitar la duplicación de lógica en distintos notebooks o etapas del pipeline
* asegurar consistencia en la medición del desempeño, especialmente en datos fuera de muestra
* facilitar la trazabilidad y reproducibilidad de los resultados

De esta manera, se establece una base sólida y homogénea para la comparación objetiva entre modelos, independientemente de su complejidad o naturaleza.




In [50]:
"""
Métricas comunes para modelos de regresión tabular en el proyecto neural_profit.

Este módulo define funciones reutilizables para evaluar predicciones de modelos
sobre targets continuos como delta_60 y delta_90.

Uso típico desde notebook:
    from stage_07_metrics import evaluate_regression_predictions

    metrics = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="ridge",
        target_name="delta_60",
    )
"""

from __future__ import annotations

from typing import Any, Dict, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def _to_1d_numpy(x: Any, name: str) -> np.ndarray:
    """
    Convierte una entrada tipo pandas/numpy/lista a un array 1D de numpy.
    """
    if isinstance(x, pd.Series):
        arr = x.to_numpy()
    elif isinstance(x, pd.DataFrame):
        if x.shape[1] != 1:
            raise ValueError(f"{name} DataFrame debe tener una sola columna.")
        arr = x.iloc[:, 0].to_numpy()
    else:
        arr = np.asarray(x)

    arr = np.ravel(arr)

    if arr.ndim != 1:
        raise ValueError(f"{name} no pudo convertirse a vector 1D.")

    return arr


def directional_accuracy(y_true: Any, y_pred: Any) -> float:
    """
    Calcula directional accuracy comparando el signo de y_true y y_pred.
    """
    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if len(y_true_arr) != len(y_pred_arr):
        raise ValueError("y_true y y_pred deben tener la misma longitud.")

    true_sign = np.sign(y_true_arr)
    pred_sign = np.sign(y_pred_arr)

    return float(np.mean(true_sign == pred_sign))


def evaluate_regression_predictions(
    y_true: Any,
    y_pred: Any,
    split_name: Optional[str] = None,
    model_name: Optional[str] = None,
    target_name: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Evalúa predicciones de regresión con métricas comunes.

    Parámetros
    ----------
    y_true : array-like
        Valores reales.
    y_pred : array-like
        Predicciones del modelo.
    split_name : str, opcional
        Nombre del split, por ejemplo: train, valid, test.
    model_name : str, opcional
        Nombre del modelo, por ejemplo: ridge, xgboost.
    target_name : str, opcional
        Nombre del target, por ejemplo: delta_60.

    Retorna
    -------
    dict
        Diccionario con métricas de evaluación.
    """
    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if len(y_true_arr) != len(y_pred_arr):
        raise ValueError("y_true y y_pred deben tener la misma longitud.")

    if len(y_true_arr) == 0:
        raise ValueError("No se puede evaluar un vector vacío.")

    mae = mean_absolute_error(y_true_arr, y_pred_arr)
    rmse = np.sqrt(mean_squared_error(y_true_arr, y_pred_arr))
    r2 = r2_score(y_true_arr, y_pred_arr)
    da = directional_accuracy(y_true_arr, y_pred_arr)

    metrics = {
        "model": model_name,
        "target": target_name,
        "split": split_name,
        "n_samples": int(len(y_true_arr)),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
        "directional_accuracy": float(da),
    }

    return metrics


def metrics_dict_to_frame(metrics: Dict[str, Any]) -> pd.DataFrame:
    """
    Convierte un diccionario de métricas en DataFrame de una fila.
    """
    return pd.DataFrame([metrics])


def print_metrics(metrics: Dict[str, Any], decimals: int = 6) -> None:
    """
    Imprime métricas de forma legible.
    """
    print("=== Regression Metrics ===")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.{decimals}f}")
        else:
            print(f"{key}: {value}")

In [51]:
COMO_USAR_METRICS = '''
from one2one_metrics import evaluate_regression_predictions, print_metrics

metrics_valid = evaluate_regression_predictions(
    y_true=y_valid_60,
    y_pred=y_pred_valid,
    split_name="valid",
    model_name="ridge",
    target_name="delta_60",
)

print_metrics(metrics_valid)
'''

# **6. Métricas**

In [52]:
from pathlib import Path
import pandas as pd


BASE_METRICS_DIR = Path("/content/drive/MyDrive/neural_profit/metrics/one2one_metrics")

REGIMES = ["premarket", "opening", "regular"]
MODELS = ["ridge", "random_forest", "xgboost", "lgbm"]


def load_regime_metrics(
    *,
    base_dir: Path = BASE_METRICS_DIR,
    regimes: list[str] = REGIMES,
    models: list[str] = MODELS,
    verbose: bool = True,
) -> dict:
    """
    Carga métricas one2one por régimen y por modelo.

    Estructura esperada:
        base_dir/
            premarket/
                one2one_ridge_all_metrics.parquet
                one2one_random_forest_all_metrics.parquet
                ...
            opening/
            regular/

    Retorna
    -------
    metrics_dict : dict
        Diccionario anidado:
        metrics_dict[regime][model] -> pd.DataFrame
    """
    metrics_dict = {}

    for regime in regimes:
        regime_dir = base_dir / regime
        metrics_dict[regime] = {}

        if verbose:
            print(f"\n==============================")
            print(f"CARGANDO RÉGIMEN: {regime}")
            print(f"Directorio: {regime_dir}")
            print(f"==============================")

        for model in models:
            file_path = regime_dir / f"one2one_{model}_all_metrics.parquet"

            if file_path.exists():
                df = pd.read_parquet(file_path).copy()
                df["regime"] = regime
                metrics_dict[regime][model] = df

                if verbose:
                    print(f"[OK] {model}: {df.shape}")
            else:
                metrics_dict[regime][model] = pd.DataFrame()

                if verbose:
                    print(f"[MISSING] {model}: {file_path.name}")

    return metrics_dict

In [53]:
metrics_by_regime = load_regime_metrics()


CARGANDO RÉGIMEN: premarket
Directorio: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/premarket
[OK] ridge: (4, 9)
[OK] random_forest: (4, 9)
[OK] xgboost: (4, 9)
[OK] lgbm: (4, 9)

CARGANDO RÉGIMEN: opening
Directorio: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/opening
[OK] ridge: (4, 9)
[OK] random_forest: (4, 9)
[OK] xgboost: (4, 9)
[OK] lgbm: (4, 9)

CARGANDO RÉGIMEN: regular
Directorio: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/regular
[OK] ridge: (4, 9)
[OK] random_forest: (4, 9)
[OK] xgboost: (4, 9)
[OK] lgbm: (4, 9)


In [54]:
from pathlib import Path
import pandas as pd


BASE_METRICS_DIR = Path("/content/drive/MyDrive/neural_profit/metrics/one2one_metrics")

REGIMES = ["premarket", "opening", "regular"]
MODELS = ["ridge", "random_forest", "xgboost", "lgbm"]


all_metrics = []

for regime in REGIMES:
    regime_dir = BASE_METRICS_DIR / regime

    for model in MODELS:
        file_path = regime_dir / f"one2one_{model}_all_metrics.parquet"

        if file_path.exists():
            df = pd.read_parquet(file_path).copy()

            # Agregar columna para identificar el régimen
            df["regime"] = regime

            all_metrics.append(df)

# Consolidar todo en una sola tabla
df_all_regimes_metrics = pd.concat(all_metrics, ignore_index=True)

# Orden opcional para visualizar mejor
df_all_regimes_metrics = df_all_regimes_metrics[
    ["regime", "model", "split", "target", "n_samples", "mae", "rmse", "r2", "directional_accuracy"]
].sort_values(
    by=["regime", "target", "split", "model"]
).reset_index(drop=True)

df_all_regimes_metrics

,regime,model,split,target,n_samples,mae,rmse,r2,directional_accuracy
0,opening,lightgbm,test,delta_60,11700,75.546386,102.355679,0.004985,0.494017
1,opening,random_forest,test,delta_60,11700,76.455132,103.077275,-0.009094,0.502650
2,opening,ridge,test,delta_60,11700,75.769697,102.694009,-0.001604,0.497778
3,opening,xgboost,test,delta_60,11700,76.047449,102.916421,-0.005947,0.498718
4,opening,lightgbm,valid,delta_60,11640,52.048377,68.792675,0.001021,0.495017
5,opening,random_forest,valid,delta_60,11640,53.294412,69.903988,-0.031515,0.501203
6,opening,ridge,valid,delta_60,11640,52.376137,68.922033,-0.002739,0.490722
7,opening,xgboost,valid,delta_60,11640,52.947895,69.592472,-0.022342,0.495962
8,opening,lightgbm,test,delta_90,11700,85.284998,113.704171,-0.003866,0.540855
9,opening,random_forest,test,delta_90,11700,86.381954,114.767816,-0.022735,0.505897


Perfecto, Gus. Aquí va un análisis **directo, técnico y sin ruido**.

---

# 🎯 1. Conclusión global (lo más importante)

👉 **No hay señal robusta en ningún régimen ni modelo**

Evidencia clara:

* **R² ≈ 0 o negativo en TEST en casi todos los casos**
* **Directional Accuracy ≈ 50% en TEST**

Esto se repite en:

* premarket
* opening
* regular
* todos los modelos

---

# 📊 2. Análisis por régimen

---

## 🟡 OPENING

### TEST (lo que importa)

* `R²`: ~0 o negativo en todos
* `DA`: 0.49 – 0.54 (muy marginal)

👉 Ejemplo:

* LightGBM delta_90 → `DA = 0.5408`
  ❗ pero:

  * no consistente entre modelos
  * no acompañado de buen R²

### Conclusión

* ligera ilusión de señal en valid/test
* **no estable → no explotable**

---

## 🔵 PREMARKET

### TEST

* `R²`: todos negativos o ~0
* `DA`: ~0.49 – 0.52

👉 peor comportamiento general

### Conclusión

👉 **ruido puro**

Esto es esperable:

* baja liquidez
* comportamiento errático

---

## 🟢 REGULAR (el más interesante)

Aquí está lo más importante del análisis.

### VALID

* algunos modelos muestran:

  * `R² > 0`
  * `DA ≈ 0.54–0.55`

👉 Ejemplo:

* LightGBM:

  * `R² ≈ 0.005`
  * `DA ≈ 0.545`

### TEST

* Ridge:

  * `R² ≈ 0.003`
  * `DA ≈ 0.51`

* resto:

  * R² negativo
  * DA ≈ 0.50

---

### 🔥 Interpretación clave

👉 **Aquí sí hay una leve señal… pero extremadamente débil**

Problema:

* no consistente entre modelos
* no estable OOS
* magnitud despreciable

---

# 📉 3. Comparación entre modelos

### Ridge

* sorprendentemente:

  * más estable
  * menos overfitting
* único que mantiene algo en TEST (aunque mínimo)

👉 **baseline fuerte**

---

### Random Forest

* consistentemente peor
* R² negativo casi siempre

👉 descartar

---

### XGBoost / LightGBM

* mejor en VALID
* empeoran en TEST

👉 típico **overfitting leve**

---

# ⚠️ 4. Insight crítico del proyecto

Esto es lo más importante de todo tu trabajo hasta ahora:

> **El problema NO es el modelo.**

Ya probaste:

* lineal
* árboles
* boosting

👉 y todos fallan igual

---

# 🧠 5. Diagnóstico real

El problema está en:

### 1. Target

* `delta_60` y `delta_90`:

  * demasiado ruidosos
  * baja relación señal/ruido

---

### 2. Features

* indicadores técnicos:

  * funcionan mal en intradía corto
  * no capturan microestructura

---

### 3. Naturaleza del problema

👉 Estás intentando predecir:

```text
movimientos intradía cortos ≈ ruido + microestructura
```

---

# 🚨 6. Conclusión formal

Puedes escribir esto (resumen técnico):

> El análisis por régimen muestra que la segmentación del mercado no produce mejoras significativas en la capacidad predictiva.
> Los modelos evaluados presentan R² cercanos a cero y directional accuracy alrededor del 50% en test, lo que indica ausencia de señal robusta.
> Se observa un leve indicio de señal en régimen regular, pero no es consistente ni generalizable.
> Por lo tanto, el problema parece estar en la definición del target y/o en la calidad del feature set, más que en la elección del modelo.

---

# 🧭 7. Qué haría ahora (orden correcto)

Este es el punto clave del proyecto.

### ❌ NO haría

* más tuning
* más modelos
* MLP / LSTM aún

---

### ✅ SÍ haría

1. **revisar target**

   * clasificación (up/down)
   * threshold
   * eventos (breakouts)

2. **revisar features**

   * volumen relativo
   * VWAP
   * order flow proxies
   * time features más fuertes

3. opcional:

   * cambiar horizonte

---

# ✔️ Conclusión final

👉 Excelente trabajo

Llegaste a una conclusión **muy valiosa y difícil de obtener**:

> **“No hay señal con esta formulación”**

Eso es progreso real en ML financiero.

---

Si quieres, el siguiente paso más potente es:

👉 rediseñar el target (ahí es donde realmente se desbloquea el proyecto).
